# Qwen2-VL MoE LoRA With Shared OCR Expert

This notebook trains one shared OCR LoRA adapter on all tasks first, then warm-starts three task-specific LoRA experts from that shared adapter before router-selected inference.

Expert ids:
- `0`: `LoRA_shared_ocr`
- `1`: `LoRA_chartqa`
- `2`: `LoRA_docvqa`
- `3`: `LoRA_textvqa`

The shared adapter is a practical PEFT-friendly approximation of the shared-expert idea: common OCR/layout behavior is learned once, then each routed task adapter starts from that shared checkpoint and specializes.

## 1. Colab Repository Setup

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git"
COLAB_REPO_DIR = Path("/content/multi-task-moe-vlm-assistant")

if Path("/content").exists():
    if not COLAB_REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "pull", "origin", "main"], check=True, cwd=COLAB_REPO_DIR)
    os.chdir(COLAB_REPO_DIR)

print(f"Current working directory: {Path.cwd()}")

## 2. Setup

In [ ]:
from pathlib import Path
from collections import Counter
import gc
import json
import os
import random
import re
import subprocess
import sys

os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

required_packages = ["qwen-vl-utils", "accelerate"]
for package_name in required_packages:
    import_name = package_name.replace("-", "_")
    try:
        __import__(import_name)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package_name], check=True)

import torch

try:
    import torchao
    torchao_version = getattr(torchao, "__version__", "0.0.0")
    major, minor, *_ = [int(part) for part in torchao_version.split(".")[:2]]
    if (major, minor) < (0, 16):
        print(f"Removing incompatible torchao=={torchao_version} before importing PEFT.")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
        for module_name in list(sys.modules):
            if module_name == "torchao" or module_name.startswith("torchao."):
                del sys.modules[module_name]
except ModuleNotFoundError:
    pass

try:
    import peft
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)

from peft import LoraConfig, PeftModel, get_peft_model
from qwen_vl_utils import process_vision_info
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from torch.utils.data import DataLoader, Dataset
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

from src.models.lora_adapter import QWEN2VL_LORA_EXPERTS
from src.routing.task_router import RouterDecision, format_router_decision

torch.__version__

## 3. Config

These settings match the single-adapter Qwen LoRA baseline notebook for a fair full comparison. Reduce `MAX_SAMPLES`, `TEST_LIMIT`, or `PRINT_LIMIT` only for debugging.

In [ ]:
METADATA_PATH = PROJECT_ROOT / "data/processed/multitask/validation.jsonl"
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"
LOCAL_CHECKPOINT_ROOT = PROJECT_ROOT / "outputs/checkpoints/qwen2vl_moe_lora_shared_ocr"
DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/multi-task-moe-vlm-assistant/checkpoints/qwen2vl_moe_lora_shared_ocr")
USE_GOOGLE_DRIVE_CHECKPOINT = True

MAX_SAMPLES = 5000
TRAIN_RATIO = 0.8
TEST_LIMIT = 1000
PRINT_LIMIT = 20
BATCH_SIZE = 1
EPOCHS = 1
LEARNING_RATE = 1e-4
GRADIENT_ACCUMULATION_STEPS = 4
SEED = 42
MAX_NEW_TOKENS = 32
ABSTENTION_THRESHOLD = 0.65
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

DATA_LIMITS = {"docvqa": 1667, "chartqa": 1667, "textvqa": 1666}
EXPECTED_TOTAL_SAMPLES = sum(DATA_LIMITS.values())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_MIXED_PRECISION = DEVICE == "cuda"
TORCH_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
DEVICE

## 4. Checkpoint Storage

In [ ]:
CHECKPOINT_ROOT = LOCAL_CHECKPOINT_ROOT

if USE_GOOGLE_DRIVE_CHECKPOINT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CHECKPOINT_ROOT = DRIVE_CHECKPOINT_ROOT
    except Exception as error:
        print(f"Google Drive checkpoint storage is unavailable: {error}")
        CHECKPOINT_ROOT = LOCAL_CHECKPOINT_ROOT

SHARED_EXPERT_ID = 0
SHARED_EXPERT_TASK = "shared_ocr"
SHARED_EXPERT_ADAPTER_NAME = "LoRA_shared_ocr"
SHARED_EXPERT_CHECKPOINT_DIR = CHECKPOINT_ROOT / SHARED_EXPERT_ADAPTER_NAME

EXPERT_CHECKPOINT_DIRS = {
    task_type: CHECKPOINT_ROOT / expert.adapter_name
    for task_type, expert in QWEN2VL_LORA_EXPERTS.items()
}
CHECKPOINT_ROOT, SHARED_EXPERT_CHECKPOINT_DIR, EXPERT_CHECKPOINT_DIRS

## 5. Prepare Data If Needed

In [ ]:
def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)


current_count = count_jsonl_lines(METADATA_PATH)
print(f"Current processed examples: {current_count}")

if current_count < EXPECTED_TOTAL_SAMPLES:
    for dataset_name, limit in DATA_LIMITS.items():
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts/prepare_data.py"),
            "--dataset",
            dataset_name,
            "--split",
            "validation",
            "--limit",
            str(limit),
            "--streaming",
        ]
        print("Running:", " ".join(command))
        subprocess.run(command, check=True, cwd=PROJECT_ROOT)

    build_command = [sys.executable, str(PROJECT_ROOT / "scripts/build_multitask_data.py"), "--split", "validation"]
    print("Running:", " ".join(build_command))
    subprocess.run(build_command, check=True, cwd=PROJECT_ROOT)
else:
    print("Processed data already exists. Skipping data preparation.")

## 6. Load Records

In [ ]:
TASK_ALIASES = {
    "chartqa": "chartqa",
    "chart_qa": "chartqa",
    "docvqa": "docvqa",
    "document_qa": "docvqa",
    "textvqa": "textvqa",
    "image_vqa": "textvqa",
}


def canonical_task_type(record: dict) -> str:
    raw_task = str(record.get("task_type") or record.get("dataset") or "").lower()
    dataset = str(record.get("dataset") or "").lower()
    return TASK_ALIASES.get(raw_task) or TASK_ALIASES.get(dataset) or "textvqa"


random.seed(SEED)
torch.manual_seed(SEED)

all_records = []
with METADATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        record["canonical_task_type"] = canonical_task_type(record)
        all_records.append(record)

all_records_by_task = {
    task_type: [record for record in all_records if record["canonical_task_type"] == task_type]
    for task_type in QWEN2VL_LORA_EXPERTS
}
for task_records in all_records_by_task.values():
    random.shuffle(task_records)

per_task_limit = max(1, MAX_SAMPLES // len(QWEN2VL_LORA_EXPERTS))
records = []
for task_type in ("chartqa", "docvqa", "textvqa"):
    records.extend(all_records_by_task[task_type][:per_task_limit])

remaining_records = [
    record
    for task_records in all_records_by_task.values()
    for record in task_records[per_task_limit:]
]
random.shuffle(remaining_records)
records.extend(remaining_records[: max(0, MAX_SAMPLES - len(records))])
random.shuffle(records)

split_index = int(len(records) * TRAIN_RATIO)
train_records = records[:split_index]
test_records = records[split_index:]

records_by_task = {
    task_type: [record for record in train_records if record["canonical_task_type"] == task_type]
    for task_type in QWEN2VL_LORA_EXPERTS
}

print("Total records:", len(records))
print("Train records:", len(train_records), Counter(record["canonical_task_type"] for record in train_records))
print("Test records:", len(test_records), Counter(record["canonical_task_type"] for record in test_records))
{task: len(items) for task, items in records_by_task.items()}

## 7. Shared Qwen Helpers

In [ ]:
def free_gpu_memory() -> None:
    for name in ["model", "optimizer", "scaler"]:
        if name in globals():
            del globals()[name]
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def load_processor():
    return AutoProcessor.from_pretrained(
        MODEL_NAME,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
    )


def load_qwen_backbone():
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=TORCH_DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    model.config.use_cache = False
    if DEVICE != "cuda":
        model.to(DEVICE)
    return model


ANSWER_FORMAT_INSTRUCTION = (
    "Answer only the direct answer. "
    "Keep all necessary words or numbers. "
    "Do not explain. "
    "If the question is unclear, malformed, not a question, or cannot be answered from readable image text, answer unanswerable."
)


def format_question_prompt(question: str) -> str:
    return f"{question}\n\n{ANSWER_FORMAT_INSTRUCTION}"


NO_ANSWER_LABELS = {
    "unanswerable",
    "no answer",
    "not a question",
    "answering does not require reading text in the image",
    "cannot be determined",
}


def normalize_for_abstention(text: str) -> str:
    return " ".join(str(text).lower().strip().split())


def reference_is_no_answer(answers: list[str]) -> bool:
    normalized_answers = [normalize_for_abstention(answer) for answer in answers]
    no_answer_votes = sum(answer in NO_ANSWER_LABELS for answer in normalized_answers)
    return no_answer_votes >= max(1, len(normalized_answers) / 2)


def clean_generated_answer(answer: str) -> str:
    cleaned = " ".join(str(answer).strip().split())
    if not cleaned:
        return cleaned

    lowered = cleaned.lower()
    if lowered.startswith("unanswerable") or lowered.startswith("not a question"):
        return "unanswerable"

    tokens = cleaned.split()
    for start in range(1, len(tokens)):
        suffix = tokens[start:]
        if len(suffix) < 3:
            continue
        numeric_like = sum(bool(re.fullmatch(r"[\d.,:%$/-]+|[a-z]*\d+[a-z]*", token.lower())) for token in suffix)
        if numeric_like / len(suffix) >= 0.8:
            return " ".join(tokens[:start]).strip(" ,.;:")

    return cleaned.strip(" ,.;:")


def build_messages(image_path: str, question: str, answer: str | None = None):
    user_content = [
        {"type": "image", "image": str(PROJECT_ROOT / image_path)},
        {"type": "text", "text": format_question_prompt(question)},
    ]
    messages = [{"role": "user", "content": user_content}]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return messages


def get_model_device(model) -> torch.device:
    return next(model.parameters()).device

## 8. Dataset and Collate Function

In [ ]:
class QwenExpertDataset(Dataset):
    def __init__(self, examples: list[dict]):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        answer = example["answers"][0] if example.get("answers") else ""
        return {
            "image_path": example["image_path"],
            "question": example["question"],
            "answer": answer,
        }


def make_collate_fn(processor):
    def collate_fn(batch):
        full_messages = [build_messages(item["image_path"], item["question"], item["answer"]) for item in batch]
        prompt_messages = [build_messages(item["image_path"], item["question"]) for item in batch]

        full_texts = [processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False) for messages in full_messages]
        prompt_texts = [processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) for messages in prompt_messages]
        image_inputs, video_inputs = process_vision_info(full_messages)

        inputs = processor(
            text=full_texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        prompt_inputs = processor(
            text=prompt_texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        labels = inputs["input_ids"].clone()

        for row_index in range(labels.shape[0]):
            prompt_length = int(prompt_inputs["attention_mask"][row_index].sum().item())
            labels[row_index, :prompt_length] = -100

        pad_token_id = processor.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        for token_id in set(processor.tokenizer.all_special_ids):
            labels[labels == token_id] = -100

        inputs["labels"] = labels
        return inputs

    return collate_fn

## 9. Train Shared OCR Expert and Task Experts

In [ ]:
def train_lora_adapter(
    adapter_label: str,
    checkpoint_dir: Path,
    examples: list[dict],
    source_adapter_dir: Path | None = None,
    expert_id: int | None = None,
) -> list[float]:
    """Train one LoRA adapter and save it.

    If source_adapter_dir is provided, the adapter starts from the shared OCR
    checkpoint before being fine-tuned for a task-specific expert.
    """

    if not examples:
        print(f"Skipping adapter={adapter_label}; no examples found.")
        return []

    expert_text = f"expert={expert_id} " if expert_id is not None else ""
    print(f"Training {expert_text}adapter={adapter_label} examples={len(examples)}")
    free_gpu_memory()

    processor = load_processor()
    model = load_qwen_backbone()
    if source_adapter_dir is None:
        lora_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            target_modules=LORA_TARGET_MODULES,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora_config)
    else:
        print(f"Warm-starting {adapter_label} from shared adapter: {source_adapter_dir}")
        model = PeftModel.from_pretrained(
            model,
            source_adapter_dir,
            adapter_name=adapter_label,
            is_trainable=True,
        )
    if DEVICE != "cuda":
        model.to(DEVICE)
    model.train()
    model.print_trainable_parameters()

    train_dataset = QwenExpertDataset(examples)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=make_collate_fn(processor),
    )

    debug_batch = next(iter(train_loader))
    valid_labels = debug_batch["labels"][debug_batch["labels"] != -100]
    print("debug label min/max:", int(valid_labels.min()), int(valid_labels.max()))
    assert int(valid_labels.min()) >= 0
    assert int(valid_labels.max()) < len(processor.tokenizer)

    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=LEARNING_RATE,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_MIXED_PRECISION)
    autocast_device = "cuda" if DEVICE == "cuda" else "cpu"
    loss_history = []
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(EPOCHS):
        for step, batch in enumerate(train_loader, start=1):
            batch = {key: value.to(get_model_device(model)) for key, value in batch.items()}
            with torch.autocast(device_type=autocast_device, dtype=torch.float16, enabled=USE_MIXED_PRECISION):
                outputs = model(**batch)
                loss = outputs.loss

            scaler.scale(loss / GRADIENT_ACCUMULATION_STEPS).backward()
            if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            loss_value = float(loss.detach().cpu())
            loss_history.append(loss_value)
            if step % 10 == 0 or step == 1:
                print(f"{expert_text}adapter={adapter_label} epoch={epoch + 1} step={step} loss={loss_value:.4f}")

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(checkpoint_dir)
    processor.save_pretrained(checkpoint_dir)
    print(f"Saved adapter={adapter_label} to {checkpoint_dir}")

    del model, optimizer, scaler
    free_gpu_memory()
    return loss_history


def train_shared_ocr_expert(examples: list[dict]) -> list[float]:
    return train_lora_adapter(
        adapter_label=SHARED_EXPERT_ADAPTER_NAME,
        checkpoint_dir=SHARED_EXPERT_CHECKPOINT_DIR,
        examples=examples,
        expert_id=SHARED_EXPERT_ID,
    )


def train_one_expert(task_type: str, examples: list[dict]) -> list[float]:
    expert = QWEN2VL_LORA_EXPERTS[task_type]
    return train_lora_adapter(
        adapter_label=expert.adapter_name,
        checkpoint_dir=EXPERT_CHECKPOINT_DIRS[task_type],
        examples=examples,
        source_adapter_dir=SHARED_EXPERT_CHECKPOINT_DIR,
        expert_id=expert.expert_id,
    )

## 10. Train Shared OCR Then All Three Experts

In [ ]:
def mean_loss(losses: list[float]) -> float | None:
    if not losses:
        return None
    return sum(losses) / len(losses)


loss_by_expert = {}
loss_by_expert[SHARED_EXPERT_TASK] = train_shared_ocr_expert(train_records)


for task_type in ("chartqa", "docvqa", "textvqa"):
    loss_by_expert[task_type] = train_one_expert(task_type, records_by_task[task_type])

all_losses = [loss for losses in loss_by_expert.values() for loss in losses]
loss_summary = {
    "shared_ocr_loss": mean_loss(loss_by_expert[SHARED_EXPERT_TASK]),
    "chartqa_loss": mean_loss(loss_by_expert["chartqa"]),
    "docvqa_loss": mean_loss(loss_by_expert["docvqa"]),
    "textvqa_loss": mean_loss(loss_by_expert["textvqa"]),
    "overall_loss": mean_loss(all_losses),
}

print("Loss summary:")
for name, value in loss_summary.items():
    print(f"{name}: {value:.4f}" if value is not None else f"{name}: None")

loss_summary

## 11. Train Learned Router

This router learns from question text and task labels. It replaces the keyword heuristic during MoE inference.

In [ ]:
router_questions = [record["question"] for record in train_records]
router_labels = [record["canonical_task_type"] for record in train_records]
label_counts = Counter(router_labels)
can_stratify = len(label_counts) == 3 and min(label_counts.values()) >= 2

train_questions, eval_questions, train_labels, eval_labels = train_test_split(
    router_questions,
    router_labels,
    test_size=0.2,
    random_state=SEED,
    stratify=router_labels if can_stratify else None,
)

router_model = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)
router_model.fit(train_questions, train_labels)

eval_predictions = router_model.predict(eval_questions)
router_accuracy = accuracy_score(eval_labels, eval_predictions)

print("Router label counts:", dict(label_counts))
print(f"Router eval accuracy: {router_accuracy:.4f}")
print(classification_report(eval_labels, eval_predictions, zero_division=0))


answerability_questions = [record["question"] for record in train_records]
answerability_labels = ["unanswerable" if reference_is_no_answer(record["answers"]) else "answerable" for record in train_records]
answerability_counts = Counter(answerability_labels)
answerability_model = None

if len(answerability_counts) == 2 and min(answerability_counts.values()) >= 2:
    answerability_model = Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)),
            ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ]
    )
    answerability_model.fit(answerability_questions, answerability_labels)

print("Answerability label counts:", dict(answerability_counts))


def should_abstain(question: str) -> tuple[bool, dict[str, float]]:
    if answerability_model is None:
        return False, {}

    probabilities = answerability_model.predict_proba([question])[0]
    classes = list(answerability_model.classes_)
    probability_by_label = {
        class_name: float(probabilities[index])
        for index, class_name in enumerate(classes)
    }
    unanswerable_probability = probability_by_label.get("unanswerable", 0.0)
    return unanswerable_probability >= ABSTENTION_THRESHOLD, probability_by_label


def learned_route_expert(question: str) -> tuple[RouterDecision, dict[str, float]]:
    probabilities = router_model.predict_proba([question])[0]
    classes = list(router_model.classes_)
    best_index = max(range(len(probabilities)), key=lambda index: probabilities[index])
    task_type = classes[best_index]
    expert = QWEN2VL_LORA_EXPERTS[task_type]
    probability_by_task = {
        class_name: float(probabilities[index])
        for index, class_name in enumerate(classes)
    }

    decision = RouterDecision(
        task_type=task_type,
        expert_id=expert.expert_id,
        adapter_name=expert.adapter_name,
        checkpoint_dir=expert.checkpoint_dir,
        confidence=float(probabilities[best_index]),
    )
    return decision, probability_by_task


router_model

## 12. Load Warm-Started MoE Adapters for Router-Based Test

In [ ]:
free_gpu_memory()
processor = load_processor()
base_model = load_qwen_backbone()

first_task = "chartqa"
model = PeftModel.from_pretrained(
    base_model,
    EXPERT_CHECKPOINT_DIRS[first_task],
    adapter_name=first_task,
)

for task_type in ("docvqa", "textvqa"):
    model.load_adapter(EXPERT_CHECKPOINT_DIRS[task_type], adapter_name=task_type)

if DEVICE != "cuda":
    model.to(DEVICE)
model.eval()
print("Loaded adapters:", list(model.peft_config.keys()))

## 13. Router-Selected MoE Inference

The router selects only the task expert (`1`, `2`, or `3`). The shared OCR behavior is inherited because each task expert was warm-started from `LoRA_shared_ocr`.

In [ ]:
def qwen_generate_with_selected_adapter(example: dict, adapter_name: str) -> str:
    model.set_adapter(adapter_name)
    messages = build_messages(example["image_path"], example["question"])
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(get_model_device(model))

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            use_cache=False,
        )

    generated_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    decoded = processor.batch_decode(generated_trimmed, skip_special_tokens=True)[0].strip()
    return clean_generated_answer(decoded)


moe_predictions = []
expert_counts = {1: 0, 2: 0, 3: 0}
test_subset = test_records[:TEST_LIMIT]

for index, example in enumerate(test_subset, start=1):
    decision, router_probs = learned_route_expert(example["question"])
    abstain, abstention_probs = should_abstain(example["question"])
    true_task = example["canonical_task_type"]
    expert_counts[decision.expert_id] = expert_counts.get(decision.expert_id, 0) + 1
    if index <= PRINT_LIMIT:
        print(format_router_decision(decision, sample_id=index, true_task_type=true_task))
        print("router_probs:", {task: round(prob, 3) for task, prob in router_probs.items()})
        print("abstention_probs:", {label: round(prob, 3) for label, prob in abstention_probs.items()})

    prediction = "unanswerable" if abstain else qwen_generate_with_selected_adapter(example, decision.task_type)
    moe_predictions.append({
        "index": index,
        "question": example["question"],
        "answers": example["answers"],
        "prediction": prediction,
        "true_task": true_task,
        "expert_id": decision.expert_id,
        "adapter_name": decision.adapter_name,
        "router_probs": router_probs,
        "abstention_probs": abstention_probs,
        "abstained": abstain,
    })
    if index <= PRINT_LIMIT:
        print("prediction:", prediction)
        print("answers:", example["answers"])

print("Evaluated samples:", len(moe_predictions))
print("Expert usage:", expert_counts)
print("Abstained samples:", sum(record["abstained"] for record in moe_predictions))
moe_predictions[:3]